# Retrieval experiments

Everything tried while building the retrieval step, scored on the tuning set (hit@3/5/10, MRR).
The baseline is a RAG built from scratch with numpy; the final pipeline lives in `src/football_rag`.

In [2]:
import json
import re

import numpy as np
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings
from langchain_openai import OpenAIEmbeddings

from football_rag import config
from football_rag.embeddings import get_embeddings
from football_rag.ingest import load_docs, split_docs, add_breadcrumbs

In [3]:
CLIENT = OpenAI(base_url=config.LMSTUDIO_URL, api_key=config.LMSTUDIO_API_KEY)
QWEN_EMBEDDING_MODEL = 'text-embedding-qwen3-embedding-0.6b'


def load_eval_set(name):
    """Load one eval set from data/eval by file name."""
    path = config.EVAL_DIR / f'{name}.json'
    with open(path, encoding='utf-8') as f:
        eval_set = json.load(f)
    return eval_set


tuning_set = load_eval_set('hand_written') + load_eval_set('tuning')
print(len(tuning_set))

48


## Chunking

In [4]:
def split_sections(text):
    """Split a Wikipedia article into (section name, body) pairs on its == headings."""
    current_name = 'Introduction'
    current_lines = []
    sections = []
    for line in text.split('\n'):
        stripped = line.strip()

        if stripped.startswith('==') and stripped.endswith('=='):
            sections.append((current_name, '\n'.join(current_lines).strip()))
            current_name = stripped.strip('= ')
            current_lines = []
        else:
            current_lines.append(line)

    sections.append((current_name, '\n'.join(current_lines).strip()))
    return sections


def fixed_chunk_text(text, chunk_size=200, overlap=40):
    """Split text into chunks of chunk_size words, with overlap words shared between neighbours."""
    words = text.split()
    start = 0
    chunks = []

    while start < len(words):
        chunk = words[start: start + chunk_size]
        chunks.append(' '.join(chunk))

        if start + chunk_size >= len(words):
            break

        start += chunk_size - overlap

    return chunks


def structure_based_chunking(text, title, chunk_size, overlap):
    """Chunk each section separately and prefix every chunk with 'title > section'."""
    sections = split_sections(text)
    all_chunks = []

    for name, body in sections:
        if not body or name in config.SKIP_SECTIONS:
            continue

        chunks = fixed_chunk_text(body, chunk_size, overlap)
        prefixed = [f'{title} > {name}: ' + c for c in chunks]
        all_chunks.extend(prefixed)

    return all_chunks

## RAG from scratch
Numpy cosine search, with switches for query rewriting, hybrid BM25 + RRF and cross-encoder reranking.

In [ ]:
def cosine(a, b):
    """Cosine similarity between two vectors."""
    dot = np.dot(a, b)
    norms = np.linalg.norm(a) * np.linalg.norm(b)
    similarity = dot / norms
    return similarity


def tokenize(text):
    """Lowercase word tokens for BM25."""
    tokens = re.findall(r'\w+', text.lower())
    return tokens


class RAG():
    """RAG built from scratch on numpy, with optional rewrite, hybrid search, and reranking."""

    def __init__(self, chunks, k=3, use_rewrite=False, use_hybrid=False, reranker_model=None, weight_cos=0.5, weight_bm25=0.5):
        self.chunks = chunks
        self.k = k
        self.use_rewrite = use_rewrite
        self.use_hybrid = use_hybrid
        self.weight_cos = weight_cos
        self.weight_bm25 = weight_bm25

        self.reranker = None
        if reranker_model is not None:
            self.reranker = CrossEncoder(reranker_model)

        tokenized = [tokenize(chunk) for chunk in chunks]
        self.bm25 = BM25Okapi(tokenized)

        prefixed = ['search_document: ' + chunk for chunk in chunks]
        response = CLIENT.embeddings.create(model=config.EMBEDDING_MODEL, input=prefixed)
        self.doc_array = np.array([item.embedding for item in response.data])


    def cosine_ranker(self, question):
        """Rank all chunks by cosine similarity to the question."""
        query = 'search_query: ' + question
        response = CLIENT.embeddings.create(model=config.EMBEDDING_MODEL, input=query)
        q_array = np.array(response.data[0].embedding)

        scores = [cosine(q_array, row) for row in self.doc_array]
        pairs = list(zip(scores, self.chunks))
        ranked = sorted(pairs, key=lambda pair: pair[0], reverse=True)
        return ranked


    def bm25_ranker(self, question):
        """Rank all chunks by BM25 keyword score."""
        scores = self.bm25.get_scores(tokenize(question))
        pairs = list(zip(scores, self.chunks))
        ranked = sorted(pairs, key=lambda pair: pair[0], reverse=True)
        return ranked


    def hybrid_ranker(self, question, rrf_k=60):
        """Fuse the cosine and BM25 rankings with weighted reciprocal rank fusion."""
        cosine_ranked = self.cosine_ranker(question)
        bm25_ranked = self.bm25_ranker(question)
        rrf_scores = {}

        for rank, (_, chunk) in enumerate(cosine_ranked, start=1):
            rrf_scores[chunk] = rrf_scores.get(chunk, 0) + self.weight_cos / (rrf_k + rank)

        for rank, (_, chunk) in enumerate(bm25_ranked, start=1):
            rrf_scores[chunk] = rrf_scores.get(chunk, 0) + self.weight_bm25 / (rrf_k + rank)

        pairs = [(score, chunk) for chunk, score in rrf_scores.items()]
        ranked = sorted(pairs, key=lambda pair: pair[0], reverse=True)
        return ranked


    def rerank(self, question, ranked, top_n=30):
        """Re-score the top_n chunks with the cross-encoder."""
        top = ranked[:top_n]
        chunks = [chunk for _, chunk in top]
        pairs = [(question, chunk) for chunk in chunks]
        scores = self.reranker.predict(pairs)

        reranked = list(zip(scores, chunks))
        reranked = sorted(reranked, key=lambda pair: pair[0], reverse=True)
        return reranked


    def rewrite(self, question):
        """Rewrite the question into a specific search query with the LLM."""
        response = CLIENT.chat.completions.create(
            model=config.CHAT_MODEL,
            messages=[
                {'role': 'system', 'content': "Rewrite the user's question as a specific search query. Expand vague references only when the question itself implies them. "
                "Do not answer the question or add facts, names, or years that are not in the question. Keep it as a question. Return only the query."},
                {'role': 'user', 'content': question}
            ],
            temperature=0
        )
        query = response.choices[0].message.content.strip()
        return query


    def rank(self, question):
        """Rank all chunks for a question, using the switches set in __init__."""
        if self.use_rewrite:
            question = self.rewrite(question)

        if self.use_hybrid:
            ranked = self.hybrid_ranker(question)
        else:
            ranked = self.cosine_ranker(question)

        if self.reranker is not None:
            ranked = self.rerank(question, ranked)

        return ranked


    def retrieve(self, question):
        """Return the top k chunks."""
        ranked = self.rank(question)
        top_k = ranked[:self.k]
        chunks = [chunk for _, chunk in top_k]
        return chunks


    def ask(self, question):
        """Retrieve, augment and generate an answer."""
        chunks = self.retrieve(question)
        context = '\n'.join(chunks)
        user_input = f'Context:\n{context}\n\nQuestion:\n{question}'

        response = CLIENT.chat.completions.create(
            model=config.CHAT_MODEL,
            messages=[
                {'role': 'system', 'content': 'Answer ONLY from context. If the context does not contain the answer, say that you do not have the answer'},
                {'role': 'user', 'content': user_input}
            ],
            temperature=0
        )
        answer = response.choices[0].message.content
        return answer


    def evaluate_retrieval(self, eval_set):
        """Hit@3/5/10 and MRR from where each question's phrase ranks; questions without a phrase are skipped."""
        ranks = []
        for item in eval_set:
            phrase = item['phrase']
            if phrase is None:
                continue

            ranked = self.rank(item['question'])
            rank = None
            for i, (_, chunk) in enumerate(ranked, start=1):
                if phrase in chunk:
                    rank = i
                    break

            ranks.append(rank)

        found = [r for r in ranks if r is not None]
        hits_3 = sum(1 for r in found if r <= 3) / len(ranks)
        hits_5 = sum(1 for r in found if r <= 5) / len(ranks)
        hits_10 = sum(1 for r in found if r <= 10) / len(ranks)
        mrr = sum(1 / r for r in found) / len(ranks)

        metrics = {'hit@3': round(hits_3, 3), 'hit@5': round(hits_5, 3), 'hit@10': round(hits_10, 3), 'MRR': round(mrr, 3)}
        return metrics

## Experiment 1: chunk size and structure

In [6]:
def load_corpus():
    """Read every article in data/corpus as (title, text) pairs."""
    corpus = []
    paths = sorted(config.CORPUS_DIR.glob('*.txt'))
    for path in paths:
        text = path.read_text(encoding='utf-8')
        corpus.append((path.stem, text))
    return corpus


def fixed_chunks(corpus, chunk_size, overlap):
    """Fixed-size chunks over each whole article, prefixed with its title."""
    all_chunks = []
    for title, text in corpus:
        chunks = fixed_chunk_text(text, chunk_size, overlap)
        prefixed = [f'{title}: ' + c for c in chunks]
        all_chunks.extend(prefixed)
    return all_chunks


def structured_chunks(corpus, chunk_size, overlap):
    """Structure-based chunks for every article."""
    all_chunks = []
    for title, text in corpus:
        chunks = structure_based_chunking(text, title, chunk_size, overlap)
        all_chunks.extend(chunks)
    return all_chunks


corpus = load_corpus()
print(len(corpus))

5
